# TextMamba3D — Unified Evaluation

通用评估 notebook，通过 `VERSION` 选择评估任何版本的 checkpoint。

| 步骤 | 操作 |
|------|------|
| 1 | 编辑 Cell 1 的 `VERSION` |
| 2 | Runtime → Run All |
| 3 | 结果自动保存到 Drive，支持跨版本对比 |


In [1]:
# ==================== CONFIG ====================
VERSION = 'v5.0'  # 'v4.4' | 'v4.5' | 'v4.6' | 'v5.0'

RUN_TTA = True          # 8-fold flip TTA (~8x slower)
PP_MIN_SIZE = 500       # ET post-processing (0 = disabled)
OVERLAP = 0.5           # sliding window overlap

PRESETS = {
    'v4.4': ('configs/textbrats_a100.yaml',      'best_v4.4.pth'),
    'v4.5': ('configs/textbrats_v7.yaml',         'best_v4.5.pth'),
    'v4.6': ('configs/textbrats_v8.yaml',         'best_v4.6.pth'),
    'v5.0': ('configs/textbrats_a100_v5.yaml',    'best_v5.0.pth'),
}
CONFIG_PATH, CKPT_NAME = PRESETS[VERSION]

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = f'{DRIVE_BASE}/checkpoints'
CKPT_PATH = f'{DRIVE_CKPT}/{CKPT_NAME}'

print(f'Version:    {VERSION}')
print(f'Config:     {CONFIG_PATH}')
print(f'Checkpoint: {CKPT_NAME}')
print(f'TTA: {RUN_TTA}  |  PP: {PP_MIN_SIZE}  |  Overlap: {OVERLAP}')


Version:    v5.0
Config:     configs/textbrats_a100_v5.yaml
Checkpoint: best_v5.0.pth
TTA: True  |  PP: 500  |  Overlap: 0.5


In [3]:
import os, shutil, zipfile, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory/ 1024**3
    print(f'GPU: {gpu} ({mem:.0f} GB)')
else:
    print('WARNING: No GPU')

# Install deps
pkgs = ['causal-conv1d', 'transformers', 'nibabel', 'tensorboard',
        'pyyaml', 'tqdm', 'scipy', 'matplotlib', 'pandas']
# All versions use PyPI mamba-ssm (Mamba2 for V5.0)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mamba-ssm'] + pkgs, check=True)

# Clone / pull repo
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print(f'Code: {REPO_DIR}')

# Extract BraTS data
DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
cases = [d for d in os.listdir(DATA_DIR) if d.startswith('BraTS20')]
print(f'Data: {len(cases)} cases')

# ET-enriched text
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
sample = sorted(cases)[0]
if not os.path.exists(os.path.join(DATA_DIR, sample, f'{sample}_et_enriched.txt')):
    if os.path.exists(ET_CACHE):
        with zipfile.ZipFile(ET_CACHE, 'r') as zf:
            zf.extractall(DATA_DIR)
        print('ET-enriched text restored')
    else:
        print('WARNING: ET-enriched text not found')
else:
    print('ET-enriched text present')

# Verify checkpoint
assert os.path.exists(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'
print(f'Checkpoint OK: {os.path.getsize(CKPT_PATH)/1024**2:.0f} MB')


In [ ]:
import subprocess, re, json, os, time

os.chdir(REPO_DIR)

# Build config matrix
configs = []
for text in [True, False]:
    tta_opts = [False, True] if RUN_TTA else [False]
    pp_opts = [False, True] if PP_MIN_SIZE > 0 else [False]
    for tta in tta_opts:
        for pp in pp_opts:
            label = 'text' if text else 'notext'
            if tta:
                label += '+TTA'
            if pp:
                label += '+PP'
            configs.append((label, text, tta, pp))

print(f'{len(configs)} configs to run')
results = {}
t0 = time.time()

for label, text, tta, pp in configs:
    print(f'--- {label} ---')
    cmd = ['python', 'evaluate_full.py',
           '--config', CONFIG_PATH,
           '--checkpoint', CKPT_PATH,
           '--split', 'test',
           '--overlap', str(OVERLAP)]
    if not text:
        cmd.append('--no-text')
    if tta:
        cmd.append('--tta')
    if pp:
        cmd.extend(['--postprocess', '--et-min-size', str(PP_MIN_SIZE)])

    log = f'eval_{VERSION}_{label.replace("+", "_")}.log'
    with open(log, 'w') as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True)

    if proc.returncode != 0:
        print(f'  FAILED (exit {proc.returncode})')
        with open(log) as f:
            lines = f.readlines()
        for line in lines[-5:]:
            print(f'  {line.rstrip()}')
        continue

    parsed = {}
    with open(log) as f:
        for line in f:
            for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean',
                        'hd95_ET', 'hd95_TC', 'hd95_WT']:
                m = re.search(rf'{key}: ([\d.]+) \+/-', line)
                if m:
                    parsed[key] = float(m.group(1))

    results[label] = parsed
    et = parsed.get('dice_ET', 0)
    tc = parsed.get('dice_TC', 0)
    wt = parsed.get('dice_WT', 0)
    mn = parsed.get('dice_mean', 0)
    print(f'  ET={et:.4f}  TC={tc:.4f}  WT={wt:.4f}  Mean={mn:.4f}')

elapsed = time.time() - t0
print(f'Done: {len(results)} configs in {elapsed/60:.0f} min')

# Save to Drive
save_dir = f'{DRIVE_BASE}/eval_results'
os.makedirs(save_dir, exist_ok=True)
save_path = f'{save_dir}/{VERSION}_results.json'
with open(save_path, 'w') as f:
    json.dump({'version': VERSION, 'config': CONFIG_PATH, 'results': results}, f, indent=2)
print(f'Saved: {save_path}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Build table
rows = []
for label, r in results.items():
    rows.append({
        'Config': f'{VERSION} {label}',
        'ET': r.get('dice_ET', 0),
        'TC': r.get('dice_TC', 0),
        'WT': r.get('dice_WT', 0),
        'Mean': r.get('dice_mean', 0),
    })
df = pd.DataFrame(rows).set_index('Config')

# Text guidance delta
if 'text' in results and 'notext' in results:
    print('Text guidance delta (text - notext):')
    for k in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean']:
        d = results['text'].get(k, 0) - results['notext'].get(k, 0)
        print(f'  {k.split("_")[1]:>4s}: {d:+.4f}')
    print()

pd.set_option('display.float_format', '{:.4f}'.format)
print(f'TextMamba3D {VERSION} — Test Set Results')
print('=' * 70)
print(df.to_string())
print()

# Chart
n = len(df)
if n == 0:
    print('No results to plot')
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(12, n * 1.8), 10),
                                   gridspec_kw={'height_ratios': [3, 1]})
    x = np.arange(n)
    w = 0.2
    for i, (col, color) in enumerate(zip(
        ['ET', 'TC', 'WT', 'Mean'],
        ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6'],
    )):
        vals = df[col].values
        bars = ax1.bar(x + (i - 1.5) * w, vals, w, label=col, color=color, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            ax1.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width() / 2, h),
                         xytext=(0, 3), textcoords='offset points',
                         ha='center', fontsize=6, rotation=90)

    ax1.set_ylabel('Dice Score')
    ax1.set_title(f'TextMamba3D {VERSION} Evaluation')
    short = [idx.replace(f'{VERSION} ', '') for idx in df.index]
    ax1.set_xticks(x)
    ax1.set_xticklabels(short, rotation=30, ha='right')
    ax1.legend(loc='lower right')
    ax1.set_ylim(0.65, 0.95)
    ax1.grid(axis='y', alpha=0.3)

    base = df['Mean'].iloc[0]
    deltas = df['Mean'].values - base
    colors = ['#2ecc71' if d >= 0 else '#e74c3c' for d in deltas]
    ax2.bar(x, deltas, 0.6, color=colors, alpha=0.8)
    ax2.axhline(y=0, color='gray', linestyle='--')
    for i, d in enumerate(deltas):
        ax2.annotate(f'{d:+.4f}', xy=(i, d),
                     xytext=(0, 3 if d >= 0 else -12),
                     textcoords='offset points', ha='center', fontsize=8)
    ax2.set_ylabel('Mean Delta')
    ax2.set_xticks(x)
    ax2.set_xticklabels(short, rotation=30, ha='right')
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{VERSION}_eval.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
import glob, json

results_dir = f'{DRIVE_BASE}/eval_results'
all_v = {}
for fp in sorted(glob.glob(f'{results_dir}/*_results.json')):
    with open(fp) as f:
        data = json.load(f)
    v = data['version']
    r = data['results']
    baseline = r.get('text', {})
    best_k = max(r, key=lambda k: r[k].get('dice_mean', 0)) if r else 'N/A'
    best = r.get(best_k, {})
    notext = r.get('notext', {})
    td = baseline.get('dice_mean', 0) - notext.get('dice_mean', 0) if notext else 0
    all_v[v] = {
        'ET': baseline.get('dice_ET', 0),
        'TC': baseline.get('dice_TC', 0),
        'WT': baseline.get('dice_WT', 0),
        'Mean': baseline.get('dice_mean', 0),
        'Best cfg': best_k,
        'Best Mean': best.get('dice_mean', 0),
        'Text delta': td,
    }

if len(all_v) > 1:
    cdf = pd.DataFrame(all_v).T
    cdf.index.name = 'Version'
    print('Cross-Version Comparison (baseline = text, no TTA, no PP)')
    print('=' * 80)
    print(cdf.to_string(float_format='{:.4f}'.format))
else:
    print(f'Only {VERSION} available. Run other versions to compare.')
    print(f'Results dir: {results_dir}/')
